# prep_imagenes_opencv

Preprocesamiento para el **CAMBIO 09** (no es un experimento: solo genera datos).

Por cada foto (`train_images/`, `test_images/`) calcula con OpenCV:
- **brillo**: media de la imagen en escala de grises (0-255);
- **nitidez**: varianza del Laplaciano (más alto = más nítida);
- **aspecto**: ancho / alto;
- **saturación**: media del canal S en HSV (0-255).

Después agrega por `PetID`: media (`mean`), máximo (`max`) y valor de la primera foto (`first`, archivo `-1.jpg`).
Guarda `train.parquet` y `test.parquet` (12 columnas, índice `PetID`).

En Kaggle: correr este notebook y agregar su salida como input del 09 con el nombre `lab2-prep-imagenes-opencv`.

In [ ]:
import glob, os
import cv2
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

def stats_foto(ruta):
    pet_id, n = os.path.basename(ruta)[:-len(".jpg")].rsplit("-", 1)     # archivo "<PetID>-<n>.jpg"
    img = cv2.imread(ruta)                                                  # BGR; None si no se puede leer
    if img is None:
        return {"PetID": pet_id, "foto": int(n)}
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    alto, ancho = gris.shape
    return {"PetID": pet_id, "foto": int(n),
            "brillo": float(gris.mean()),
            "nitidez": float(cv2.Laplacian(gris, cv2.CV_64F).var()),
            "aspecto": ancho / alto,
            "saturacion": float(hsv[:, :, 1].mean())}

VARIABLES = ["brillo", "nitidez", "aspecto", "saturacion"]

def por_mascota(carpeta):
    rutas = glob.glob(os.path.join(carpeta, "*.jpg"))
    fotos = pd.DataFrame(Parallel(n_jobs=-1, batch_size=256)(delayed(stats_foto)(r) for r in rutas))
    print(carpeta, "-> fotos:", len(fotos), "| no se pudieron leer:", fotos["brillo"].isna().sum())
    g = fotos.groupby("PetID")[VARIABLES]
    media = g.mean().add_prefix("img_cv_mean_")
    maximo = g.max().add_prefix("img_cv_max_")
    primera = fotos[fotos["foto"] == 1].set_index("PetID")[VARIABLES].add_prefix("img_cv_first_")
    return media.join(maximo).join(primera)

In [ ]:
# Local: guarda con la misma ruta que tendría en Kaggle como input. En Kaggle: en la carpeta de trabajo (salida).
SALIDA = "." if os.path.exists("/kaggle") else "../input/lab2-prep-imagenes-opencv"
os.makedirs(SALIDA, exist_ok=True)
for conjunto in ["train", "test"]:
    df = por_mascota(f"../input/petfinder-adoption-prediction/{conjunto}_images")
    df.to_parquet(os.path.join(SALIDA, f"{conjunto}.parquet"))
    print(conjunto, df.shape)
df.describe().T.round(2)